In [ ]:
from pathlib import Path

RESULTS_ROOT = Path(
    "/tmp/wwpgd/results/experiments/level_00/multiplier_20"
)
ANALYSIS_DIR = RESULTS_ROOT / "analysis"

# Single-Level Multiseed AdamW vs AdamW+WW-PGD Analysis

This notebook discovers paired runs, audits available logging, analyzes token-matched loss curves, paired terminal validation losses, runtime/compute fields, and WW-PGD projection diagnostics. It uses repository-emitted files (`manifest.json`, `metrics.csv`, `wwpgd_projection.csv`, `environment.json`, and related manifests) via reusable helpers in `wwgpt.analysis`.

In [ ]:
import sys, json, math, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display, Markdown


def find_repo_root(start: Path) -> Path:
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / 'pyproject.toml').exists() and (p / 'src' / 'wwgpt').exists():
            return p
    raise RuntimeError('Could not locate repository root from current working directory')

REPO_ROOT = find_repo_root(Path.cwd())
SRC = REPO_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from wwgpt.analysis import discover_experiment_runs, normalize_metrics, terminal_results, align_curves, paired_curve_differences, summary

ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Repository root: {REPO_ROOT}')
print(f'Results root: {RESULTS_ROOT}')
print(f'Analysis output: {ANALYSIS_DIR}')
if not RESULTS_ROOT.exists():
    raise FileNotFoundError(f'RESULTS_ROOT does not exist: {RESULTS_ROOT}. Change only the first cell, then Run All.')

## Result Discovery and Run Inventory

In [ ]:
runs = discover_experiment_runs(RESULTS_ROOT)
if not runs:
    raise RuntimeError(f'No pair_* directories found under {RESULTS_ROOT}')

def scalar(row, names):
    for n in names:
        if n in row and pd.notna(row[n]):
            return row[n]
    return np.nan

def run_inventory_row(r):
    art = r['artifacts']; man = art.get('manifest.json', {}); comp = art.get('run_complete.json', {}); env = art.get('environment.json', {})
    m = normalize_metrics(art.get('metrics.csv', pd.DataFrame()))
    last = m.sort_values('step').tail(1) if 'step' in m and not m.empty else m.tail(1)
    minrow = pd.DataFrame()
    if 'validation_loss' in m:
        minrow = m.loc[pd.to_numeric(m['validation_loss'], errors='coerce').idxmin():pd.to_numeric(m['validation_loss'], errors='coerce').idxmin()]
    elapsed = scalar(last.iloc[0], ['elapsed_seconds','elapsed_time']) if not last.empty else np.nan
    proj_total = pd.to_numeric(m.get('projection_seconds', m.get('projection_overhead', pd.Series(dtype=float))), errors='coerce').sum() if not m.empty else np.nan
    param_report = man.get('parameter_report', {}) if isinstance(man.get('parameter_report', {}), dict) else {}
    records = len(m)
    status = 'completed' if art.get('run_complete.json') else ('missing' if r['run_dir'] is None else 'incomplete')
    events = art.get('events.jsonl', [])
    exit_status = 'complete' if any(e.get('event') == 'complete' for e in events) or comp else ('unknown' if r['run_dir'] else 'missing')
    return {
        'pair identifier': r['pair_id'], 'seed': r['seed'], 'optimizer': r['optimizer'], 'run directory': str(r['run_dir']) if r['run_dir'] else '',
        'selection note': r['selection_note'], 'completion status': status, 'exit status': exit_status,
        'number of logged records': records, 'number of optimizer steps': man.get('optimizer_steps', comp.get('step', scalar(last.iloc[0], ['step']) if not last.empty else np.nan)),
        'tokens seen': scalar(last.iloc[0], ['tokens_seen','tokens_processed']) if not last.empty else np.nan,
        'target tokens': man.get('realized_tokens', man.get('requested_tokens', np.nan)),
        'tokens per parameter': (man.get('realized_tokens', np.nan) / param_report.get('total_parameters', np.nan)) if param_report.get('total_parameters') else np.nan,
        'final training loss': scalar(last.iloc[0], ['train_loss']) if not last.empty else np.nan,
        'final validation loss': scalar(last.iloc[0], ['validation_loss','val_loss']) if not last.empty else np.nan,
        'minimum validation loss': pd.to_numeric(m.get('validation_loss', pd.Series(dtype=float)), errors='coerce').min() if 'validation_loss' in m else np.nan,
        'step or token count at minimum validation loss': scalar(minrow.iloc[0], ['tokens_seen','tokens_processed','step']) if not minrow.empty else np.nan,
        'elapsed wall-clock seconds': elapsed,
        'training seconds': np.nan,
        'evaluation seconds': np.nan,
        'projection seconds': proj_total,
        'estimated FLOPs': man.get('estimated_flops', np.nan),
        'tokens per second': scalar(last.iloc[0], ['tokens_per_second']) if not last.empty else np.nan,
        'parameter count': param_report.get('total_parameters', np.nan),
        'git commit': man.get('git_commit', env.get('git_commit', np.nan)),
    }

inventory = pd.DataFrame([run_inventory_row(r) for r in runs])
inventory.to_csv(ANALYSIS_DIR / 'run_inventory.csv', index=False)
display(inventory)

## Logging and Schema Audit

In [ ]:
print('Files loaded per run:')
for r in runs:
    print(f"\n{r['pair_id']} / {r['optimizer']} -> {r['run_dir']}")
    for p in r['artifacts'].get('files_loaded', []):
        print(f'  {p}')

print('\nDiscovered columns / fields by file type:')
file_fields = {}
for r in runs:
    for name, obj in r['artifacts'].items():
        if name == 'files_loaded' or name == 'run_dir': continue
        if isinstance(obj, pd.DataFrame):
            fields = list(obj.columns)
        elif isinstance(obj, dict):
            fields = list(obj.keys())
        elif isinstance(obj, list):
            fields = sorted({k for rec in obj if isinstance(rec, dict) for k in rec})
        else:
            fields = ['<text>']
        file_fields.setdefault(name, set()).update(fields)
for name, fields in sorted(file_fields.items()):
    print(f'{name}: {sorted(fields)}')

coverage_specs = {
 'step': [('metrics.csv','step'), ('wwpgd_projection.csv','step')],
 'tokens_seen': [('metrics.csv','tokens_seen'), ('metrics.csv','tokens_processed')],
 'unique_tokens_seen': [('metrics.csv','unique_tokens_seen')],
 'target_tokens': [('manifest.json','realized_tokens'), ('manifest.json','requested_tokens'), ('data_manifest.json','realized_tokens')],
 'tokens_per_parameter': [('manifest.json','parameter_report')],
 'train_loss': [('metrics.csv','train_loss')],
 'validation_loss': [('metrics.csv','validation_loss'), ('metrics.csv','val_loss')],
 'learning_rate': [('metrics.csv','learning_rate')],
 'gradient_norm': [('metrics.csv','gradient_norm')],
 'parameter_norm': [('metrics.csv','parameter_norm')],
 'update_norm': [('metrics.csv','update_norm')],
 'elapsed_seconds': [('metrics.csv','elapsed_seconds'), ('metrics.csv','elapsed_time')],
 'training_seconds': [('metrics.csv','training_seconds')],
 'evaluation_seconds': [('metrics.csv','evaluation_seconds')],
 'estimated_flops': [('manifest.json','estimated_flops')],
 'tokens_per_second': [('metrics.csv','tokens_per_second')],
 'projection event number': [('wwpgd_projection.csv','event'), ('wwpgd_projection.csv','projection_event')],
 'projection token position': [('wwpgd_projection.csv','tokens_seen'), ('wwpgd_projection.csv','tokens_processed')],
 'projection duration': [('wwpgd_projection.csv','projection_runtime'), ('metrics.csv','projection_overhead')],
 'projected layer': [('wwpgd_projection.csv','layer_name')],
 'alpha before': [('wwpgd_projection.csv','alpha_before')],
 'alpha after': [('wwpgd_projection.csv','alpha_after')],
 'trace-log before': [('wwpgd_projection.csv','trace_log_before')],
 'trace-log after': [('wwpgd_projection.csv','trace_log_after')],
 'detX before': [('wwpgd_projection.csv','detX_before'), ('wwpgd_projection.csv','detX_val')],
 'detX after': [('wwpgd_projection.csv','detX_after')],
 'relative projection norm': [('wwpgd_projection.csv','relative_frobenius_change'), ('wwpgd_projection.csv','relative_frobenius_weight_change')],
 'projection skip reason': [('wwpgd_projection.csv','warning'), ('wwpgd_projection.csv','skip_reason')],
 'initialization hash': [('manifest.json','initialization_hash'), ('initialization_hash.txt','<text>')],
 'data-order seed or manifest hash': [('manifest.json','data_hash'), ('manifest.json','corpus_hash'), ('data_manifest.json','corpus_hash')],
 'dataset identifier': [('manifest.json','dataset_name'), ('data_manifest.json','dataset_name'), ('data_manifest.json','dataset')],
 'tokenizer identifier': [('manifest.json','tokenizer_hash'), ('tokenizer_manifest.json','tokenizer_hash'), ('tokenizer_manifest.json','tokenizer')],
 'git commit': [('manifest.json','git_commit'), ('environment.json','git_commit')],
 'hardware information': [('environment.json','device'), ('environment.json','processor'), ('environment.json','operating_system')],
 'PyTorch version': [('environment.json','torch_version')],
}

def has_field(art, file, field):
    obj = art.get(file)
    if obj is None: return False
    if isinstance(obj, pd.DataFrame): return field in obj.columns
    if isinstance(obj, dict): return field in obj
    if isinstance(obj, list): return any(isinstance(x, dict) and field in x for x in obj)
    return file == 'initialization_hash.txt' and field == '<text>'

cov_rows=[]
valid_runs = [r for r in runs if r['run_dir'] is not None]
for field, specs in coverage_specs.items():
    hits=[]
    for r in valid_runs:
        hit = [(f,c) for f,c in specs if has_field(r['artifacts'], f, c)]
        if hit: hits.append((r, hit[0]))
    if not hits: status='missing'; source=''
    elif len(hits) == len(valid_runs): status='available'; source=', '.join(sorted({f'{f}:{c}' for _,(f,c) in hits}))
    else: status='partially available'; source=', '.join(sorted({f'{f}:{c}' for _,(f,c) in hits}))
    cov_rows.append({'field': field, 'status': status, 'source file and column': source})
coverage = pd.DataFrame(cov_rows)
coverage.to_csv(ANALYSIS_DIR / 'logging_coverage.csv', index=False)
display(coverage)
missing = coverage.loc[coverage['status']!='available','field'].tolist()
if missing:
    warnings.warn('Some optional logging fields are not fully available: ' + ', '.join(missing))

## Training and Validation Curves

In [ ]:
metrics_by_run=[]
for r in runs:
    m=normalize_metrics(r['artifacts'].get('metrics.csv', pd.DataFrame()))
    if not m.empty:
        m=m.copy(); m['optimizer']=r['optimizer']; m['seed']=r['seed']; m['pair_id']=r['pair_id']; metrics_by_run.append(m)

def plot_loss(y_col, ylabel, fname, title):
    curves=[]; labels=[]
    plt.figure(figsize=(9,5))
    x_col = 'tokens_seen' if all('tokens_seen' in m for m in metrics_by_run if y_col in m) else 'step'
    for opt,color in [('adamw','tab:blue'),('adamw_wwpgd','tab:orange')]:
        opt_curves=[]
        for m in metrics_by_run:
            if m['optimizer'].iloc[0]==opt and y_col in m and x_col in m:
                d=m[[x_col,y_col]].apply(pd.to_numeric, errors='coerce').dropna().sort_values(x_col)
                if len(d)>=2:
                    plt.plot(d[x_col], d[y_col], color=color, alpha=.25, lw=1)
                    opt_curves.append(d)
        grid, vals=align_curves(opt_curves, x_col, y_col)
        if len(grid):
            mean=vals.mean(axis=0); se=vals.std(axis=0, ddof=1)/np.sqrt(vals.shape[0]) if vals.shape[0]>1 else np.zeros_like(mean)
            band=stats.t.ppf(.975, vals.shape[0]-1)*se if vals.shape[0]>1 else se
            plt.plot(grid, mean, color=color, lw=2.5, label=f'{opt} mean (n={vals.shape[0]})')
            plt.fill_between(grid, mean-band, mean+band, color=color, alpha=.15, label=f'{opt} 95% CI' if vals.shape[0]>1 else f'{opt} band unavailable n=1')
    plt.xlabel('tokens seen' if x_col=='tokens_seen' else 'optimizer step'); plt.ylabel(ylabel); plt.title(title); plt.grid(True, alpha=.3); plt.legend(); plt.tight_layout()
    path=ANALYSIS_DIR/fname; plt.savefig(path, dpi=160); plt.show(); print(f'Saved {path}')
plot_loss('validation_loss','validation loss','validation_loss_vs_tokens.png', 'Validation loss vs token progress')
plot_loss('train_loss','training loss','training_loss_vs_tokens.png', 'Training loss vs token progress')

## Paired Terminal Results

In [ ]:
terminal = terminal_results(runs)
terminal.to_csv(ANALYSIS_DIR/'paired_terminal_results.csv', index=False)
print('Final is defined as the last recorded validation evaluation after sorting by tokens_seen when available, otherwise by optimizer step. Negative WW-PGD minus AdamW values favor WW-PGD.')
display(terminal)
complete = terminal.dropna(subset=['adamw_final_validation_loss','adamw_wwpgd_final_validation_loss']) if not terminal.empty else terminal
if not complete.empty:
    diffs=complete['wwpgd_minus_adamw_final_validation_loss']
    n=len(diffs); sd=diffs.std(ddof=1) if n>1 else 0.0; se=sd/math.sqrt(n) if n else np.nan; ci=stats.t.ppf(.975,n-1)*se if n>1 else 0.0
    report=pd.DataFrame([{
        'complete pairs':n, 'mean AdamW final validation loss':complete['adamw_final_validation_loss'].mean(), 'standard deviation of AdamW final validation loss':complete['adamw_final_validation_loss'].std(ddof=1),
        'mean WW-PGD final validation loss':complete['adamw_wwpgd_final_validation_loss'].mean(), 'standard deviation of WW-PGD final validation loss':complete['adamw_wwpgd_final_validation_loss'].std(ddof=1),
        'mean paired difference':diffs.mean(), 'median paired difference':diffs.median(), 'standard deviation of paired differences':sd, 'standard error of paired difference':se,
        '95 percent CI low':diffs.mean()-ci, '95 percent CI high':diffs.mean()+ci,
        'WW-PGD lower final validation loss seeds':int((diffs<0).sum()), 'ties':int((diffs==0).sum()), 'AdamW lower final validation loss seeds':int((diffs>0).sum())}])
    display(report)
    plt.figure(figsize=(6,4))
    for _, row in complete.iterrows(): plt.plot(['AdamW','AdamW+WW-PGD'], [row['adamw_final_validation_loss'], row['adamw_wwpgd_final_validation_loss']], marker='o', alpha=.7)
    plt.ylabel('final validation loss'); plt.title('Paired final validation loss by seed'); plt.grid(True, axis='y', alpha=.3); plt.tight_layout(); plt.savefig(ANALYSIS_DIR/'paired_final_validation_loss.png', dpi=160); plt.show()
    plt.figure(figsize=(7,4)); plt.axhline(0,color='black',lw=1); plt.bar(complete['seed'].astype(str), diffs); plt.ylabel('WW-PGD minus AdamW final validation loss'); plt.xlabel('seed'); plt.title('Paired final validation-loss difference\nnegative values favor WW-PGD'); plt.grid(True, axis='y', alpha=.3); plt.tight_layout(); plt.savefig(ANALYSIS_DIR/'paired_final_loss_difference.png', dpi=160); plt.show()
else:
    print('No complete AdamW/WW-PGD pairs with validation loss are available.')

## Paired Difference During Training

In [ ]:
pairs=[]
for pair_id, g in pd.DataFrame([{'pair_id':r['pair_id'],'optimizer':r['optimizer'],'i':i} for i,r in enumerate(runs)]).groupby('pair_id'):
    idx={row.optimizer: int(row.i) for row in g.itertuples()}
    if {'adamw','adamw_wwpgd'}.issubset(idx):
        a=normalize_metrics(runs[idx['adamw']]['artifacts'].get('metrics.csv', pd.DataFrame()))
        w=normalize_metrics(runs[idx['adamw_wwpgd']]['artifacts'].get('metrics.csv', pd.DataFrame()))
        if not a.empty and not w.empty: pairs.append((a,w))
x_col='tokens_seen' if all('tokens_seen' in d for pair in pairs for d in pair) else 'step'
grid, diffs=paired_curve_differences(pairs, x_col, 'validation_loss')
if len(grid):
    plt.figure(figsize=(9,5));
    for d in diffs: plt.plot(grid,d,color='gray',alpha=.25)
    mean=diffs.mean(axis=0); se=diffs.std(axis=0,ddof=1)/np.sqrt(diffs.shape[0]) if diffs.shape[0]>1 else np.zeros_like(mean); band=stats.t.ppf(.975,diffs.shape[0]-1)*se if diffs.shape[0]>1 else se
    plt.plot(grid,mean,color='black',lw=2.5,label=f'mean paired difference (n={diffs.shape[0]})'); plt.fill_between(grid,mean-band,mean+band,color='black',alpha=.15); plt.axhline(0,color='red',lw=1)
    plt.xlabel('tokens seen' if x_col=='tokens_seen' else 'optimizer step'); plt.ylabel('WW-PGD validation loss minus AdamW'); plt.title('Paired validation-loss difference during training\nnegative values favor WW-PGD; positive values favor AdamW'); plt.grid(True,alpha=.3); plt.legend(); plt.tight_layout(); plt.savefig(ANALYSIS_DIR/'paired_validation_loss_difference_vs_tokens.png', dpi=160); plt.show()
else: print('Paired validation-loss curves could not be aligned without extrapolation.')

## Compute and Runtime Efficiency

In [ ]:
compute_cols=['seed','pair identifier','optimizer','final validation loss','elapsed wall-clock seconds','training seconds','evaluation seconds','projection seconds','tokens per second','estimated FLOPs']
compute = inventory[[c for c in compute_cols if c in inventory.columns]].copy()
compute.to_csv(ANALYSIS_DIR/'compute_runtime_by_run.csv', index=False)
display(compute)
if 'projection seconds' in compute and 'elapsed wall-clock seconds' in compute:
    compute['projection overhead percentage']=100*pd.to_numeric(compute['projection seconds'],errors='coerce')/pd.to_numeric(compute['elapsed wall-clock seconds'],errors='coerce')
summary_rows=[]
for opt,g in compute.groupby('optimizer'):
    row={'optimizer':opt, 'runs':len(g)}
    for c in compute.select_dtypes(include='number').columns:
        row[f'mean {c}']=g[c].mean()
    summary_rows.append(row)
compute_summary=pd.DataFrame(summary_rows); compute_summary.to_csv(ANALYSIS_DIR/'compute_runtime_summary.csv', index=False); display(compute_summary)
if {'final validation loss','elapsed wall-clock seconds'}.issubset(compute.columns):
    plt.figure(figsize=(6,4))
    for opt,g in compute.groupby('optimizer'): plt.scatter(g['elapsed wall-clock seconds'], g['final validation loss'], label=opt)
    plt.xlabel('wall-clock seconds'); plt.ylabel('final validation loss'); plt.title('Final validation loss vs wall-clock time'); plt.grid(True,alpha=.3); plt.legend(); plt.tight_layout(); plt.savefig(ANALYSIS_DIR/'final_validation_loss_vs_wall_clock_time.png', dpi=160); plt.show()
if metrics_by_run and all('elapsed_seconds' in m for m in metrics_by_run):
    plt.figure(figsize=(8,5))
    for opt,color in [('adamw','tab:blue'),('adamw_wwpgd','tab:orange')]:
        for m in metrics_by_run:
            if m['optimizer'].iloc[0]==opt and 'validation_loss' in m:
                plt.plot(m['elapsed_seconds'],m['validation_loss'],color=color,alpha=.25)
    plt.xlabel('wall-clock seconds'); plt.ylabel('validation loss'); plt.title('Validation loss vs wall-clock time'); plt.grid(True,alpha=.3); plt.tight_layout(); plt.savefig(ANALYSIS_DIR/'validation_loss_vs_wall_clock_time.png', dpi=160); plt.show()
if 'tokens per second' in compute and compute['tokens per second'].notna().any():
    plt.figure(figsize=(6,4)); compute.boxplot(column='tokens per second', by='optimizer'); plt.suptitle(''); plt.title('Tokens per second by optimizer'); plt.tight_layout(); plt.savefig(ANALYSIS_DIR/'tokens_per_second_by_optimizer.png', dpi=160); plt.show()
else: print('Tokens per second was not logged, so throughput analysis is unavailable for this run.')
if 'projection overhead percentage' in compute and compute['projection overhead percentage'].notna().any():
    ww=compute[compute['optimizer']=='adamw_wwpgd']; plt.figure(figsize=(7,4)); plt.bar(ww['seed'].astype(str), ww['projection overhead percentage']); plt.xlabel('seed'); plt.ylabel('projection overhead (% total runtime)'); plt.title('Projection overhead by seed'); plt.tight_layout(); plt.savefig(ANALYSIS_DIR/'projection_overhead_by_seed.png', dpi=160); plt.show()
else: print('Projection duration was not logged, so projection-overhead analysis is unavailable for this run.')
# Threshold timing: only thresholds both arms reach.
threshold_rows=[]
if pairs:
    mins=[]
    for a,w in pairs:
        if 'validation_loss' in a and 'validation_loss' in w: mins.append(max(a['validation_loss'].min(), w['validation_loss'].min()))
    if mins:
        thresholds=np.linspace(max(mins), max(min(m['validation_loss'].max() for pair in pairs for m in pair if 'validation_loss' in m), max(mins)), 3)
        for ti,t in enumerate(sorted(set(np.round(thresholds,6)))):
            for pi,(a,w) in enumerate(pairs):
                for opt,d in [('adamw',a),('adamw_wwpgd',w)]:
                    hit=d[pd.to_numeric(d['validation_loss'],errors='coerce')<=t]
                    if not hit.empty:
                        threshold_rows.append({'pair_index':pi,'optimizer':opt,'threshold':t,'tokens_to_threshold':hit.iloc[0].get('tokens_seen',np.nan),'seconds_to_threshold':hit.iloc[0].get('elapsed_seconds',np.nan)})
threshold_df=pd.DataFrame(threshold_rows); threshold_df.to_csv(ANALYSIS_DIR/'time_to_common_validation_loss_thresholds.csv', index=False)
if not threshold_df.empty:
    display(threshold_df); plt.figure(figsize=(8,4));
    for opt,g in threshold_df.groupby('optimizer'): plt.plot(g.groupby('threshold')['seconds_to_threshold'].mean(), marker='o', label=opt)
    plt.xlabel('validation-loss threshold'); plt.ylabel('mean seconds to threshold'); plt.title('Time to common validation-loss thresholds'); plt.grid(True,alpha=.3); plt.legend(); plt.tight_layout(); plt.savefig(ANALYSIS_DIR/'time_to_common_validation_loss_thresholds.png', dpi=160); plt.show()
else: print('No common validation-loss thresholds were reached by both optimizer arms.')

## WW-PGD Projection Diagnostics

In [ ]:
proj=[]
for r in runs:
    p=r['artifacts'].get('wwpgd_projection.csv', pd.DataFrame())
    if not p.empty:
        p=p.copy(); p['seed']=r['seed']; p['pair_id']=r['pair_id'];
        if 'tokens_seen' not in p and 'step' in p:
            target = inventory[(inventory['pair identifier']==r['pair_id']) & (inventory['optimizer']==r['optimizer'])]['target tokens']
            steps = inventory[(inventory['pair identifier']==r['pair_id']) & (inventory['optimizer']==r['optimizer'])]['number of optimizer steps']
            if len(target) and len(steps) and pd.notna(target.iloc[0]) and pd.notna(steps.iloc[0]) and steps.iloc[0]:
                p['tokens_seen']=p['step']*float(target.iloc[0])/float(steps.iloc[0])
        proj.append(p)
proj_df=pd.concat(proj, ignore_index=True) if proj else pd.DataFrame()
proj_df.to_csv(ANALYSIS_DIR/'wwpgd_projection_records.csv', index=False)
if proj_df.empty:
    print('No WW-PGD projection records were logged.')
else:
    display(proj_df.head())
    numeric=[c for c in proj_df.select_dtypes(include='number').columns]
    proj_summary=pd.DataFrame([{'metric':c, **summary(proj_df[c])} for c in numeric]); proj_summary.to_csv(ANALYSIS_DIR/'wwpgd_projection_summary.csv', index=False); display(proj_summary)
    if {'tokens_seen','step'}.issubset(proj_df.columns):
        plt.figure(figsize=(7,4)); plt.scatter(proj_df['tokens_seen'], proj_df['step'], alpha=.7); plt.xlabel('tokens seen at projection'); plt.ylabel('training step'); plt.title('Projection events over token progress'); plt.grid(True,alpha=.3); plt.tight_layout(); plt.savefig(ANALYSIS_DIR/'projection_events_over_token_progress.png', dpi=160); plt.show()
    if 'projection_runtime' in proj_df:
        plt.figure(figsize=(7,4)); plt.plot(np.arange(len(proj_df)), proj_df['projection_runtime'], marker='o', ls='none'); plt.xlabel('projection record'); plt.ylabel('projection duration seconds'); plt.title('Projection duration by event record'); plt.grid(True,alpha=.3); plt.tight_layout(); plt.savefig(ANALYSIS_DIR/'projection_duration_by_event.png', dpi=160); plt.show()
    else: print('Projection duration was not logged in projection records.')
    rel_col = 'relative_frobenius_change' if 'relative_frobenius_change' in proj_df else ('relative_frobenius_weight_change' if 'relative_frobenius_weight_change' in proj_df else None)
    if rel_col and 'layer_name' in proj_df:
        piv=proj_df.pivot_table(index='step',columns='layer_name',values=rel_col,aggfunc='mean'); piv.plot(figsize=(9,5), title='Relative projection norm by layer and event'); plt.ylabel(rel_col); plt.grid(True,alpha=.3); plt.tight_layout(); plt.savefig(ANALYSIS_DIR/'relative_projection_norm_by_layer_event.png', dpi=160); plt.show()
    if {'alpha_before','alpha_after'}.issubset(proj_df.columns):
        plt.figure(figsize=(5,5)); plt.scatter(proj_df['alpha_before'], proj_df['alpha_after'], alpha=.6); lim=[min(proj_df['alpha_before'].min(),proj_df['alpha_after'].min()), max(proj_df['alpha_before'].max(),proj_df['alpha_after'].max())]; plt.plot(lim,lim,color='black',lw=1); plt.xlabel('alpha before'); plt.ylabel('alpha after'); plt.title('Alpha before vs after projection'); plt.grid(True,alpha=.3); plt.tight_layout(); plt.savefig(ANALYSIS_DIR/'alpha_before_vs_after.png', dpi=160); plt.show()
    else: print('Alpha before/after was not logged.')
    if {'trace_log_before','trace_log_after'}.issubset(proj_df.columns):
        plt.figure(figsize=(5,5)); plt.scatter(proj_df['trace_log_before'], proj_df['trace_log_after']); plt.xlabel('trace-log before'); plt.ylabel('trace-log after'); plt.title('Trace-log before vs after'); plt.tight_layout(); plt.savefig(ANALYSIS_DIR/'trace_log_before_vs_after.png', dpi=160); plt.show()
    else: print('Trace-log before/after was not logged, so trace-log projection diagnostics are unavailable.')
    loss_cols={'loss_before_projection','loss_after_projection'}
    if loss_cols.issubset(proj_df.columns):
        proj_df['projection_loss_change']=proj_df['loss_after_projection']-proj_df['loss_before_projection']; proj_df.plot(y='projection_loss_change', figsize=(7,4), title='Projection-induced loss change'); plt.tight_layout(); plt.savefig(ANALYSIS_DIR/'projection_induced_loss_change.png', dpi=160); plt.show()
    else: print('Immediate loss before/after projection was not logged, so projection-induced loss-change analysis is unavailable.')

## Optimizer Summary

In [ ]:
supported = {
 'optimizer':'optimizer', 'completed runs':'completion status', 'mean final validation loss':'final validation loss', 'standard deviation of final validation loss':'final validation loss',
 'mean minimum validation loss':'minimum validation loss', 'mean elapsed time':'elapsed wall-clock seconds', 'mean training time':'training seconds', 'mean projection time':'projection seconds',
 'mean projection overhead percentage':'projection overhead percentage', 'mean tokens per second':'tokens per second', 'mean estimated FLOPs':'estimated FLOPs'}
inv = inventory.copy()
if 'projection overhead percentage' not in inv and {'projection seconds','elapsed wall-clock seconds'}.issubset(inv.columns):
    inv['projection overhead percentage']=100*pd.to_numeric(inv['projection seconds'],errors='coerce')/pd.to_numeric(inv['elapsed wall-clock seconds'],errors='coerce')
rows=[]
for opt,g in inv.groupby('optimizer'):
    row={'optimizer':opt, 'completed runs':int((g['completion status']=='completed').sum())}
    if g['final validation loss'].notna().any(): row['mean final validation loss']=g['final validation loss'].mean(); row['standard deviation of final validation loss']=g['final validation loss'].std(ddof=1)
    for outcol, incol in [('mean minimum validation loss','minimum validation loss'),('mean elapsed time','elapsed wall-clock seconds'),('mean training time','training seconds'),('mean projection time','projection seconds'),('mean projection overhead percentage','projection overhead percentage'),('mean tokens per second','tokens per second'),('mean estimated FLOPs','estimated FLOPs')]:
        if incol in g and g[incol].notna().any(): row[outcol]=pd.to_numeric(g[incol], errors='coerce').mean()
    rows.append(row)
optimizer_summary=pd.DataFrame(rows); optimizer_summary.to_csv(ANALYSIS_DIR/'optimizer_summary.csv', index=False); display(optimizer_summary)

## Preliminary Interpretation

In [ ]:
lines=[]
if 'complete' in globals() and not complete.empty:
    mean_a=complete['adamw_final_validation_loss'].mean(); mean_w=complete['adamw_wwpgd_final_validation_loss'].mean(); diffs=complete['wwpgd_minus_adamw_final_validation_loss']; n=len(diffs)
    lower='AdamW+WW-PGD' if mean_w < mean_a else ('AdamW' if mean_a < mean_w else 'neither optimizer (tie)')
    sd=diffs.std(ddof=1) if n>1 else 0.0; se=sd/math.sqrt(n) if n else np.nan; ci=stats.t.ppf(.975,n-1)*se if n>1 else 0.0
    ci_low, ci_high=diffs.mean()-ci, diffs.mean()+ci
    direction='WW-PGD' if diffs.mean()<0 else ('AdamW' if diffs.mean()>0 else 'neither optimizer')
    agree=int((diffs<0).sum() if diffs.mean()<0 else (diffs>0).sum() if diffs.mean()>0 else (diffs==0).sum())
    lines.append(f'{lower} has the lower mean final validation loss at the recorded terminal evaluation.')
    lines.append(f'The mean paired difference (WW-PGD minus AdamW) is {diffs.mean():.6g}, so the average direction favors {direction}; {agree} of {n} paired seeds agree with that direction.')
    lines.append(f'The paired 95% confidence interval is [{ci_low:.6g}, {ci_high:.6g}], which {"includes" if ci_low <= 0 <= ci_high else "does not include"} zero. This should not be overinterpreted with only {n} seeds.')
else:
    lines.append('No complete paired terminal validation-loss comparison was available.')
if 'compute' in globals() and 'projection overhead percentage' in compute and compute.loc[compute['optimizer']=='adamw_wwpgd','projection overhead percentage'].notna().any():
    overhead=compute.loc[compute['optimizer']=='adamw_wwpgd','projection overhead percentage'].mean(); lines.append(f'WW-PGD adds measurable logged projection runtime overhead averaging {overhead:.3g}% of elapsed runtime among WW-PGD runs.')
else:
    lines.append('Projection runtime overhead could not be fully assessed because projection duration or elapsed runtime was not logged for all required runs.')
lines.append('The practical compute conclusion should be compared with the token-matched loss conclusion above; if runtime, throughput, or FLOP fields are missing, no substitute metric is used.')
incomplete=inventory[inventory['completion status']!='completed']
lines.append(f'{len(incomplete)} run inventory rows appear incomplete, missing, or anomalous.' if len(incomplete) else 'No discovered run inventory rows appear incomplete based on run_complete.json.')
important_missing=coverage.loc[coverage['status']=='missing','field'].tolist()
lines.append('Important missing logging fields include: ' + (', '.join(important_missing) if important_missing else 'none from the audit list.'))
lines.append('This is a five-seed comparison at one model size and one token budget. It can establish a preliminary paired optimizer effect at this operating point, but it does not establish a scaling-law result or show how the effect changes with model size or training compute.')
display(Markdown('\n\n'.join(f'- {line}' for line in lines)))